# Position weight matrix: motif logo and scan

*bioinformatics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/pwm_motif_scan)

Aligned binding sites of a transcription factor are counted into a $(m, 4)$ matrix, turned into frequencies with a pseudocount and into log-odds scores against the background. The information content gives a sequence logo, and one einsum over unfolded windows scores both strands of a long sequence at once; hits are called with a p-value taken from shuffled copies of the sequence.

**Vector form**

$$P_j(a) = \frac{C_j(a) + \beta}{N + 4\beta},\qquad W_j(a) = \log_2\frac{P_j(a)}{q(a)},\qquad \mathrm{IC}_j = \sum_a P_j(a)\log_2\frac{P_j(a)}{q(a)},\qquad \mathrm{score}(n) = \sum_{j=0}^{m-1} W_j(s_{n+j})$$

**Index (tensor) form**

$$C_{ja} = X_{sja} \qquad P_{ja} = \frac{C_{ja} + \beta}{N + 4\beta} \qquad \mathrm{IC}_j = P_{ja}\log_2\frac{P_{ja}}{q_a} \qquad \mathrm{score}_{tn} = U_{naj}\,W_{tja},\quad U_{naj} = X_{n+j,\,a}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $X_{sja}$ | `one_hot(sites)` | (N, m, 4) | training site $s$, column $j$, letter $a$ ∈ (A, C, G, T) |
| $C_{ja} = X_{sja}$ | `einsum('sja->ja', X)` | (m, 4) | repeated $s$ ⇒ summed: letter counts in every column |
| $P_{ja}$ | `(C + β) / (N + 4β)` | (m, 4) | frequencies with pseudocount $\beta$; each row sums to 1 and no entry is 0 |
| $W_{ja}$ | `log2(P / q)` | (m, 4) | log-odds against the background $q_a$ (bits): > 0 preferred, < 0 avoided |
| $\mathrm{IC}_j$ | `einsum('ja,ja->j', P, log2(P / q))` | (m,) | repeated $a$ ⇒ summed: bits per column, between 0 and 2 for a uniform background; the logo height |
| $W_{tja}$ | `stack([W, W.flip(0).flip(1)])` | (2, m, 4) | $t = 0$ plus strand, $t = 1$ reverse complement (columns reversed, A↔T, C↔G) |
| $U_{naj}$ | `X.unfold(-2, m, 1)` | (L−m+1, 4, m) | every window of $m$ letters of the long sequence |
| $\mathrm{score}_{tn}$ | `einsum('...naj,sja->...sn', U, W2)` | (2, L−m+1) | repeated $j$, $a$ ⇒ summed: both strands in one call; the same code scores all shuffles $(R, 2, L-m+1)$ |
| $p_{tn}$ | `p_values(score, null)` | (2, L−m+1) | fraction of shuffled-sequence scores at least as high |

A PWM assumes that each position of a binding site contributes independently, so the score of a window is a sum of one number per letter: an inner product of the one-hot window with $W$. The pseudocount $\beta$ keeps letters never seen in a few training sites from getting a score of $-\infty$; it matters less the more sites you have. The information content of a column is how much the motif narrows down the letter there, from 0 bits (anything goes) to 2 bits (always the same letter), and the logo stacks the letters in proportion to $P_{ja}$. The sites here are synthetic: 12 bp with a core resembling the TATA box (consensus TATAWAWR, with W = A or T and R = A or G) and two fixed letters on each side, so that the motif carries enough bits to be found. A short degenerate motif alone carries too little information: in a long sequence random windows score as high as real sites, which is why a real TATA box is hard to find by its PWM alone. The p-value comes from scanning shuffled copies of the sequence (same letters, no motif); with $2(L-m+1)$ windows a cut at $p$ lets through about $2(L-m+1)\,p$ random hits, and the smallest p-value you can resolve is $1/(1 + \#\text{null scores})$. Try fewer training sites, a weaker motif or a looser cut and watch planted sites go missing or false hits appear.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/pwm_motif_scan/main.py
"""Main loop: training sites → PWM → scan a long sequence (both strands) → empirical p-values → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    gen = torch.Generator().manual_seed(params["seed"])
    M = tensors.true_motif(params["strength"], dev)  # (m, 4) hidden motif
    q = tensors.background(dev)  # (4,)
    train = tensors.sample_sites(M, params["n_train"], gen)  # (N, m) aligned binding sites
    pwm = solve.build_pwm(train, params["pseudo"], q)

    seq, starts, strands = tensors.sequence_with_sites(M, params["length"], params["n_sites"], gen)
    scores = solve.scan_both_strands(seq, pwm["W"])  # (2, n)
    null = solve.scan_both_strands(tensors.shuffles(seq, params["shuffles"], gen), pwm["W"])  # (R, 2, n)
    p = equation.p_values(scores, null)
    p_max = 10 ** params["log10_p"]
    hits = solve.call_hits(p, p_max)
    found = solve.match(hits, starts, strands)

    m, n = M.shape[0], scores.shape[1]
    print(f"PWM from {params['n_train']} sites of length {m}   ·   pseudocount β = {params['pseudo']}")
    print("consensus      " + "".join(tensors.BASES[a] for a in pwm["P"].argmax(dim=1).tolist()))
    print("IC (bits)      " + " ".join(f"{v:.2f}" for v in pwm["IC"].tolist()) + f"   total {pwm['IC'].sum():.1f}")
    print(f"max score {pwm['W'].max(dim=1).values.sum():.2f} bits   ·   {2 * n:,} windows scanned (both strands)")
    s_thr = solve.score_threshold(null, p_max)
    print(
        f"null: {null.numel():,} scores from {params['shuffles']} shuffles "
        f"(smallest possible p = {1 / (1 + null.numel()):.1e})"
    )
    print(f"p ≤ {p_max:.0e} ⇔ score > {s_thr:.2f} bits: {len(hits)} hits, about {p_max * 2 * n:.1f} expected by chance")
    for i in range(len(starts)):
        s0, sd = int(starts[i]), int(strands[i])
        print(
            f"  planted {s0:>6} {'+' if sd > 0 else '−'}  score {scores[0 if sd > 0 else 1, s0]:6.2f}  "
            f"p = {p[0 if sd > 0 else 1, s0]:.1e}  {'found' if found[i] else 'missed'}"
        )
    false_hits = [h for h in hits if not any(h == (int(a), int(b)) for a, b in zip(starts, strands))]
    print(f"found {int(found.sum())}/{len(starts)} planted sites   ·   {len(false_hits)} other hits")
    return plot.figure(pwm, scores, null, s_thr, p_max, starts, strands, hits, out_dir / "pwm_motif_scan.png")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/pwm_motif_scan/equation.py
r"""Sequence → tensor translation for a position weight matrix (PWM).

Aligned training sites become a one-hot tensor X_{sja} (site s, column j, letter a). Then

    C_{ja} = X_{sja}                               (s summed: letter counts per column)
    P_{ja} = (C_{ja} + β) / (N + 4β)               (frequencies with a pseudocount β, rows sum to 1)
    W_{ja} = log2(P_{ja} / q_a)                    (log-odds against the background q, in bits)
    IC_j   = P_{ja} log2(P_{ja} / q_a)              (a summed: information content, 0..2 bits for uniform q)

Scanning a sequence X_{na}: unfold it into every window of m letters, U_{naj} = X_{n+j, a}, and score both strands
in one einsum with the stacked matrices W_{sja} (s = 0: the PWM, s = 1: its reverse complement),

    score_{sn} = U_{naj} W_{sja}                    (j and a summed)

The reverse complement reads the other strand: column order reversed and A↔T, C↔G swapped. With the letters ordered
A, C, G, T, the complement of letter a is 3 - a, so it is just W flipped along both axes.
"""

import torch


def counts(X: torch.Tensor) -> torch.Tensor:
    """C_{ja} = X_{sja}  (s summed).   (N, m, 4) → (m, 4)"""
    return torch.einsum("sja->ja", X)


def frequencies(C: torch.Tensor, pseudo: float) -> torch.Tensor:
    """P_{ja} = (C_{ja} + β) / (N + 4β): never exactly 0, so the log-odds stay finite.   (m, 4)"""
    N = C.sum(dim=1, keepdim=True)  # (m, 1) number of sites
    return (C + pseudo) / (N + 4 * pseudo)


def log_odds(P: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    """W_{ja} = log2(P_{ja} / q_a): positive for letters the site prefers, negative for avoided ones.   (m, 4)"""
    return torch.log2(P / q)


def information_content(P: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    """IC_j = P_{ja} log2(P_{ja} / q_a)  (a summed): bits per column; with q uniform this is 2 - entropy.   (m,)"""
    return torch.einsum("ja,ja->j", P, torch.log2(P / q))


def reverse_complement_matrix(W: torch.Tensor) -> torch.Tensor:
    """W^rc_{ja} = W_{m-1-j, 3-a}: the same site read on the other strand.   (m, 4)"""
    return W.flip(0).flip(1)


def windows(X: torch.Tensor, m: int) -> torch.Tensor:
    """U_{…naj} = X_{…, n+j, a}: every window of m letters.   (…, L, 4) → (…, L-m+1, 4, m), no copy."""
    return X.unfold(-2, m, 1)


def scan(X: torch.Tensor, W_both: torch.Tensor) -> torch.Tensor:
    r"""score_{…sn} = U_{…naj} W_{sja}  (j and a summed).   X (…, L, 4), W_both (2, m, 4) → (…, 2, L-m+1)."""
    U = windows(X, W_both.shape[1])
    return torch.einsum("...naj,sja->...sn", U, W_both)


def p_values(scores: torch.Tensor, null: torch.Tensor) -> torch.Tensor:
    """p = (1 + #{null ≥ score}) / (1 + #null): how often a shuffled sequence scores at least as high.  Same shape."""
    null_sorted = null.flatten().sort().values
    n_below = torch.searchsorted(null_sorted, scores.contiguous(), right=False)  # null values < score
    return (1 + null_sorted.numel() - n_below) / (1 + null_sorted.numel())


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/pwm_motif_scan/tensors.py
"""Build the concrete tensors: a synthetic motif, aligned training sites, and a long sequence with planted sites.

Bases are numbered A = 0, C = 1, G = 2, T = 3, so the complement of base a is 3 - a.
"""

import torch

BASES = "ACGT"
IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "W": "AT", "R": "AG", "S": "CG"}
# A synthetic 12-bp site: a TATA-box-like core (TATAWAWR, W = A/T, R = A/G) with two preferred letters on each
# side. It is not a palindrome, so a site on the plus strand does not also score high on the minus strand.
CONSENSUS = "GCTATAWAWRAG"


def true_motif(strength: float, device: torch.device) -> torch.Tensor:
    """The hidden column probabilities M_{ja} used to generate sites.   (m, 4), rows sum to 1.

    In column j the letters allowed by the IUPAC code share the probability `strength`; the others share the rest.
    """
    rows = []
    for code in CONSENSUS:
        allowed = [a for a, b in enumerate(BASES) if b in IUPAC[code]]
        row = torch.full((4,), (1 - strength) / (4 - len(allowed)))
        row[allowed] = strength / len(allowed)
        rows.append(row)
    return torch.stack(rows).to(device)


def background(device: torch.device) -> torch.Tensor:
    """q_a: background letter frequencies of the random sequence (uniform here).   (4,)"""
    return torch.full((4,), 0.25, device=device)


def sample_sites(M: torch.Tensor, n: int, gen: torch.Generator) -> torch.Tensor:
    """n aligned sites drawn column by column from M, as letter indices.   (n, m)"""
    return torch.multinomial(M.cpu(), n, replacement=True, generator=gen).T.to(M.device)


def one_hot(seq: torch.Tensor) -> torch.Tensor:
    """X_{…a} = 1 where the letter is a.   (…,) → (…, 4)"""
    return torch.nn.functional.one_hot(seq, num_classes=4).float()


def reverse_complement(seq: torch.Tensor) -> torch.Tensor:
    """Read the other strand: reverse the order and swap A↔T, C↔G (a → 3 - a).   (…, m) → (…, m)"""
    return 3 - seq.flip(-1)


def sequence_with_sites(
    M: torch.Tensor, length: int, n_sites: int, gen: torch.Generator
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """A random sequence (L,) with n_sites motif instances; every second one is planted on the minus strand.

    Returns seq (L,), the start positions (n_sites,) and the strands (n_sites,) with +1 / -1.
    """
    m = M.shape[0]
    seq = torch.randint(0, 4, (length,), generator=gen).to(M.device)
    slot = length // n_sites  # one site somewhere inside each slot, so they never overlap
    starts = torch.arange(n_sites) * slot + torch.randint(0, slot - m, (n_sites,), generator=gen)
    strands = torch.tensor([1 if i % 2 == 0 else -1 for i in range(n_sites)])
    sites = sample_sites(M, n_sites, gen)  # (n_sites, m)
    for i in range(n_sites):
        site = sites[i] if strands[i] > 0 else reverse_complement(sites[i])
        seq[starts[i] : starts[i] + m] = site
    return seq, starts.to(M.device), strands.to(M.device)


def shuffles(seq: torch.Tensor, n: int, gen: torch.Generator) -> torch.Tensor:
    """n random permutations of seq: same letter composition, no motif left.   (L,) → (n, L)"""
    order = torch.argsort(torch.rand(n, seq.shape[0], generator=gen), dim=1).to(seq.device)
    return seq[order]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/pwm_motif_scan/solve.py
"""Build the PWM from training sites, scan a sequence on both strands and call hits with an empirical p-value."""

import torch

from . import equation
from .tensors import one_hot


def build_pwm(sites: torch.Tensor, pseudo: float, q: torch.Tensor) -> dict[str, torch.Tensor]:
    """sites (N, m) letter indices → counts, frequencies, log-odds (all (m, 4)) and information content (m,)."""
    C = equation.counts(one_hot(sites))
    P = equation.frequencies(C, pseudo)
    return {"C": C, "P": P, "W": equation.log_odds(P, q), "IC": equation.information_content(P, q)}


def scan_both_strands(seq: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    """Scores (…, 2, L-m+1) of every window of seq (…, L): row 0 the plus strand, row 1 the minus strand."""
    W_both = torch.stack([W, equation.reverse_complement_matrix(W)])  # (2, m, 4)
    return equation.scan(one_hot(seq), W_both)


def call_hits(p: torch.Tensor, p_max: float) -> list[tuple[int, int]]:
    """(start, strand ±1) of every window with p ≤ p_max, sorted by position.   p: (2, n)"""
    strand_row, start = torch.nonzero(p <= p_max, as_tuple=True)
    hits = [(int(n), 1 if int(s) == 0 else -1) for s, n in zip(strand_row, start)]
    return sorted(hits)


def match(hits: list[tuple[int, int]], starts: torch.Tensor, strands: torch.Tensor, tol: int = 0) -> torch.Tensor:
    """found_i = True if planted site i (start, strand) is among the hits.   (n_sites,) bool"""
    found = [any(abs(h - int(s0)) <= tol and hs == int(sd) for h, hs in hits) for s0, sd in zip(starts, strands)]
    return torch.tensor(found)


def score_threshold(null: torch.Tensor, p_max: float) -> float:
    """The score a window must beat to reach p ≤ p_max: the (K+1)-th highest null score, K = p_max (1 + #null) - 1."""
    null_sorted = null.flatten().sort(descending=True).values
    K = int(p_max * (1 + null_sorted.numel())) - 1
    return float(null_sorted[max(K, 0)]) if K >= 0 else float("inf")


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/pwm_motif_scan/plot.py
"""Static figure: sequence logo, log-odds matrix, null score distribution and the scan along the sequence."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.font_manager import FontProperties
from matplotlib.patches import PathPatch
from matplotlib.textpath import TextPath
from matplotlib.transforms import Affine2D

from problems._common.plotting import save_png

from .tensors import BASES

LETTER_COLORS = {"A": "#16a34a", "C": "#2563eb", "G": "#f59e0b", "T": "#dc2626"}
PLUS, MINUS = "#2563eb", "#ea580c"
FONT = FontProperties(family="DejaVu Sans", weight="bold")


def figure(
    pwm: dict[str, torch.Tensor],
    scores: torch.Tensor,
    null: torch.Tensor,
    s_thr: float,
    p_max: float,
    starts: torch.Tensor,
    strands: torch.Tensor,
    hits: list[tuple[int, int]],
    path: Path,
) -> Path:
    P, W, IC = pwm["P"].cpu().numpy(), pwm["W"].cpu().numpy(), pwm["IC"].cpu().numpy()
    sc, nl = scores.cpu().numpy(), null.flatten().cpu().numpy()
    m = P.shape[0]

    fig = plt.figure(figsize=(9.6, 5.6))
    grid = fig.add_gridspec(2, 3, width_ratios=[1.35, 1, 1], height_ratios=[1, 1.05])
    ax_logo = fig.add_subplot(grid[0, 0])
    ax_w = fig.add_subplot(grid[0, 1])
    ax_h = fig.add_subplot(grid[0, 2])
    ax_s = fig.add_subplot(grid[1, :])

    # --- sequence logo: column height = information content, letters scaled by P_ja · IC_j ---
    for j in range(m):
        y = 0.0
        for a in np.argsort(P[j]):  # smallest at the bottom, the preferred letter on top
            h = P[j, a] * IC[j]
            if h > 0.01:
                _letter(ax_logo, BASES[a], j + 1, y, h)
            y += h
    ax_logo.set_xlim(0.5, m + 0.5)
    ax_logo.set_ylim(0, 2)
    ax_logo.set_xticks(range(1, m + 1))
    ax_logo.tick_params(axis="x", labelsize=7)
    ax_logo.set_ylabel("bits")
    ax_logo.set_xlabel("position $j$", labelpad=1)
    ax_logo.set_title(f"sequence logo, total {IC.sum():.1f} bits", fontsize=10)
    ax_logo.grid(False)

    # --- log-odds matrix W_ja ---
    lim = np.abs(W).max()
    im = ax_w.imshow(W.T, cmap=plt.get_cmap("RdBu_r", 12), vmin=-lim, vmax=lim, aspect="auto")
    ax_w.set_yticks(range(4), list(BASES))
    ax_w.set_xticks(range(0, m, 2), [str(j + 1) for j in range(0, m, 2)])
    ax_w.set_xlabel("position $j$", labelpad=1)
    ax_w.set_title(r"log-odds $W_{ja}$ (bits)", fontsize=10)
    ax_w.grid(False)
    fig.colorbar(im, ax=ax_w, fraction=0.06, pad=0.03)

    # --- null distribution from shuffled sequences vs the real scan ---
    bins = np.linspace(min(nl.min(), sc.min()), max(nl.max(), sc.max()) + 0.5, 50)
    ax_h.hist(nl, bins=bins, color="#9ca3af", density=True, label="shuffled")
    ax_h.hist(sc.ravel(), bins=bins, histtype="step", color="#111827", density=True, label="sequence")
    ax_h.axvline(s_thr, color="#dc2626", ls="--", lw=1.2)
    ax_h.set_yscale("log")
    ax_h.set_xlabel("score (bits)", labelpad=1)
    ax_h.set_title(f"null scores, cut at p = {p_max:.0e}", fontsize=10)
    ax_h.legend(fontsize=7, loc="upper left")

    # --- scan along the sequence: plus strand up, minus strand mirrored down ---
    x = np.arange(sc.shape[1])
    ax_s.plot(x, np.clip(sc[0], 0, None), color=PLUS, lw=0.6, label="+ strand")
    ax_s.plot(x, -np.clip(sc[1], 0, None), color=MINUS, lw=0.6, label="− strand")
    for sign in (1, -1):
        ax_s.axhline(sign * s_thr, color="#dc2626", ls="--", lw=1)
    top = max(sc.max(), s_thr) * 1.25
    for s0, sd in zip(starts.tolist(), strands.tolist()):
        ax_s.plot([s0], [sd * top * 0.93], "v" if sd > 0 else "^", color="#374151", ms=6)
    for h, hs in hits:
        ax_s.plot([h], [hs * sc[0 if hs > 0 else 1, h]], "o", ms=7, mfc="none", mew=1.5, color="#111827")
    ax_s.set_xlim(0, len(x))
    ax_s.set_ylim(-top, top)
    ax_s.set_xlabel("window start $n$ (bp)", labelpad=1)
    ax_s.set_ylabel("score (bits, > 0 shown)")
    ax_s.set_title("scan: ▼▲ planted sites, ○ hits above the threshold (dashed)", fontsize=10)
    ax_s.legend(fontsize=7.5, loc="upper right", ncol=2)

    fig.tight_layout()
    return save_png(fig, path)


def _letter(ax: plt.Axes, char: str, x: float, y: float, height: float) -> None:
    """Draw one logo letter filling the box [x - 0.45, x + 0.45] × [y, y + height]."""
    tp = TextPath((0, 0), char, size=1, prop=FONT)
    box = tp.get_extents()
    t = Affine2D().translate(-box.x0, -box.y0).scale(0.9 / box.width, height / box.height).translate(x - 0.45, y)
    ax.add_patch(PathPatch(t.transform_path(tp), color=LETTER_COLORS[char], lw=0))


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_train': 30,
    'strength': 0.9,
    'pseudo': 0.5,
    'length': 5000,
    'n_sites': 8,
    'log10_p': -4.0,
    'shuffles': 20,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.pwm_motif_scan")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.pwm_motif_scan.main")
problem_dir = Path("problems/bioinformatics/pwm_motif_scan")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
